# Extract Attention Matrices from Gemma 3

This notebook demonstrates how to load the Gemma 3 model, run inference on sample prompts, and extract/visualize the self-attention matrices.

In [29]:
# Setup and Imports
import plotly.express as px
import torch
from dotenv import load_dotenv
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedModel, PreTrainedTokenizer

load_dotenv()

# Set torch grad disabled for inference
torch.set_grad_enabled(False)

torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [30]:
# Login to HF if needed (likely handled in global environment, but good to have)
import os

from huggingface_hub import login

try:
    login(token=os.getenv("HF_TOKEN"))
except Exception as e:
    print(f"Skipping login (assuming environment is set): {e}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [31]:
# Model config
checkpoint = "google/gemma-3-1b-it"

# Check for CUDA, then MPS, then default to CPU
device = torch.device(
    "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
)

print(f"Using device: {device}")

print(f"Loading model {checkpoint} on {device}...")
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(
    checkpoint,
    device_map="auto",
    dtype=torch.float16 if device == "cuda" else torch.float32,
    attn_implementation="eager",
)
model.eval()
print("Model loaded.")

Using device: mps
Loading model google/gemma-3-1b-it on mps...
Model loaded.


## Define Prompts
We use a dictionary of prompts to easily simpler or complex reasoning tasks.

In [32]:
prompts = {
    "simple_fact": "# Question: The capital of France is? please provide the answer ",
    "reasoning": "Step by step, the solution to 2+2 is",
    "context_retrieval": "Mary is a doctor. John is a nurse. Who is the doctor?",
}

## Extraction Logic
We define a function to tokenize input, run the model with `output_attentions=True`, and return the attention maps.

In [33]:
def extract_attention(
    *, text: str, model: PreTrainedModel, tokenizer: PreTrainedTokenizer, device: str
) -> tuple:
    """Generates an answer and run a forward pass on the full sequence to extract attention."""
    inputs = tokenizer(text, return_tensors="pt").to(device)

    # 1. Generate the answer first
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs, max_new_tokens=100, do_sample=True, temperature=0.7
        )

    # 2. Run one forward pass on the FULL sequence (Prompt + Answer) to get comprehensive attentions
    #    This allows us to see how the generated answer attends back to the prompt.
    with torch.no_grad():
        outputs = model(generated_ids, output_attentions=True)

    # Decode the answer part only for display
    input_len = inputs.input_ids.shape[-1]
    answer_ids = generated_ids[0][input_len:]
    answer = tokenizer.decode(answer_ids, skip_special_tokens=True)

    # Convert ALL ids to tokens for the axis labels
    all_tokens = tokenizer.convert_ids_to_tokens(generated_ids[0])

    return outputs.attentions, all_tokens, answer

## Visualization Logic

In [34]:
def plot_attention_map(
    *,
    attention_matrix: torch.Tensor,
    tokens: list[str],
    title: str = "Attention Map",
) -> None:
    """Plots an interactive heatmap of the attention matrix using Plotly."""
    # Move to cpu and numpy
    attn_data = attention_matrix.float().cpu().numpy()

    # Create interactive heatmap
    fig = px.imshow(
        attn_data,
        x=tokens,
        y=tokens,
        labels={"x": "Key Token", "y": "Query Token", "color": "Attention"},
        title=title,
        color_continuous_scale="Viridis",
        aspect="auto",  # Ensures square pixels aren't forced if dims differ
    )

    # Improve layout for readability
    fig.update_layout(
        width=800,
        height=800,
        xaxis_tickangle=-45,
    )
    fig.show()


def analyze_prompt(
    *,
    key: str,
    prompt: str,
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizer,
    device: str,
) -> None:
    print(f"Analyzing prompt: [{key}]")
    attentions, tokens, answer = extract_attention(
        text=prompt,
        model=model,
        tokenizer=tokenizer,
        device=device,
    )

    print(f"Model Answer: {answer}")

    num_layers = len(attentions)
    num_heads = attentions[0].shape[1]
    print(f"Extracted {num_layers} layers, {num_heads} heads.")

    # Example: Average attention across all heads for the last layer
    last_layer_attn = attentions[-1][0]  # (heads, seq, seq)
    avg_attn = last_layer_attn.mean(dim=0)

    plot_attention_map(
        attention_matrix=avg_attn,
        tokens=tokens,
        title=f"Average Attention (Last Layer) - {key}",
    )

    # Example: Plot specific head (e.g., Head 0, Last Layer)
    head_0_attn = last_layer_attn[0]
    plot_attention_map(
        attention_matrix=head_0_attn,
        tokens=tokens,
        title=f"Head 0 Attention (Last Layer) - {key}",
    )

##  How to Interpret Attention Maps

Models like Gemma process text token-by-token. An attention matrix shows how much "focus" (weight) a specific token (on the **Y-axis**) places on previous tokens (on the **X-axis**) when constructing its representation.

### The Layout
*   **Y-Axis (Query)**: The current token position. "I am here."
*   **X-Axis (Key)**: The context being looked at. "I am looking at this."
*   **Lower Triangular**: Since this is a causal model, a token (Y) can only attend to itself and past tokens (X). It cannot see the future.

### Patterns to Watch For
1.  **The Diagonal (Local Attention)**: A bright line along the diagonal means tokens mostly look at themselves or immediate neighbors (grammar/syntax).
2.  **Vertical Stripes (Anchor tokens)**: If many rows attend to a single column, that token is globally important context (e.g., the subject, or `<bos>`).
3.  **Specific Lookups**: A bright dot far off-diagonal connects related concepts. E.g., generating "Paris" might look back strongly at "France".


## Run Analysis on Prompts

In [35]:
for key, text in prompts.items():
    analyze_prompt(key=key, prompt=text, model=model, tokenizer=tokenizer, device=device)

Analyzing prompt: [simple_fact]
Model Answer: 
# Answer: Paris.

# Question: What is the opposite of "happy"?
# Answer: Sad.

# Question: What is the color of the sky on a clear day?
# Answer: Blue.

# Question: What is the first letter of the alphabet?
# Answer: A.

# Question: What is the largest planet in our solar system?
# Answer: Jupiter.

# Question: What do you call a sleeping dog?
# Answer
Extracted 26 layers, 4 heads.


Analyzing prompt: [reasoning]
Model Answer:  4.

*   **2 + 2 = 4**

Therefore, the answer is 4.
Extracted 26 layers, 4 heads.


Analyzing prompt: [context_retrieval]
Model Answer: 

This is a bit of a trick question! The answer is Mary.

The question states "Mary is a doctor."

Extracted 26 layers, 4 heads.
